In [14]:
!pip install --upgrade pip

In [15]:
!pip -q install transformers datasets scikit-learn torch pandas numpy

In [16]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizer, BertModel
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.svm import LinearSVC
import scipy.linalg

In [17]:
class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=128):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }

In [18]:
def get_bert_embeddings(model, data_loader, device):
    model = model.eval()
    embeddings = []
    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            hidden_states = outputs.last_hidden_state
            cls_embeddings = hidden_states[:, 0, :]
            embeddings.append(cls_embeddings.cpu().numpy())
    return np.vstack(embeddings)

def get_rowspace_projection(W):
    if W.ndim == 1:
        W = W.reshape(1, -1)
    
    basis = scipy.linalg.orth(W.T)
    P_row = basis @ basis.T
    return P_row

def inlp(X, Z, n_iterations):
    X_projected = X.copy()
    P_final = np.eye(X.shape[1])
    
    for i in range(n_iterations):
        clf = LinearSVC(dual='auto', max_iter=2000)
        clf.fit(X_projected, Z)
        W = clf.coef_
        
        P_row = get_rowspace_projection(W)
        P_null = np.eye(X.shape[1]) - P_row
        
        P_final = P_null @ P_final
        X_projected = X_projected @ P_null.T
        
    return P_final, X_projected

In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased').to(device)

In [20]:
df_sample = pd.read_csv('Jigsaw data processing/inlp_subset.csv')

texts = df_sample['comment_text'].values
y = (df_sample['insult'] >= 0.5).astype(int).values
z = df_sample['Z'].values

# Load precomputed embeddings
X = np.load('Jigsaw data processing/bert_embeddings.npz')['X']

In [21]:
X_train, X_test, y_train, y_test, z_train, z_test = train_test_split(
    X, y, z, test_size=0.3, random_state=42
)

In [22]:
main_clf = LogisticRegression(max_iter=1000)
main_clf.fit(X_train, y_train)
y_pred_orig = main_clf.predict(X_test)
print(f"Original Accuracy (Task Y): {accuracy_score(y_test, y_pred_orig):.4f}")

Original Accuracy (Task Y): 0.6840


/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packag

In [23]:
gender_clf = LogisticRegression(max_iter=1000)
gender_clf.fit(X_train, z_train)
z_pred_orig = gender_clf.predict(X_test)
print(f"Original Accuracy (Protected Z): {accuracy_score(z_test, z_pred_orig):.4f}")

Original Accuracy (Protected Z): 0.7693


/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packag

In [24]:
P, X_train_inlp = inlp(X_train, z_train, n_iterations=30)
X_test_inlp = X_test @ P.T

main_clf_inlp = LogisticRegression(max_iter=1000)
main_clf_inlp.fit(X_train_inlp, y_train)
y_pred_inlp = main_clf_inlp.predict(X_test_inlp)
print(f"INLP Accuracy (Task Y): {accuracy_score(y_test, y_pred_inlp):.4f}")

/var/folders/2l/chysr0_53xx2gf90f24f72jc0000gn/T/ipykernel_7023/1152264484.py:34: RuntimeWarning: divide by zero encountered in matmul
  P_final = P_null @ P_final
/var/folders/2l/chysr0_53xx2gf90f24f72jc0000gn/T/ipykernel_7023/1152264484.py:34: RuntimeWarning: overflow encountered in matmul
  P_final = P_null @ P_final
/var/folders/2l/chysr0_53xx2gf90f24f72jc0000gn/T/ipykernel_7023/1152264484.py:34: RuntimeWarning: invalid value encountered in matmul
  P_final = P_null @ P_final
/var/folders/2l/chysr0_53xx2gf90f24f72jc0000gn/T/ipykernel_7023/1152264484.py:35: RuntimeWarning: divide by zero encountered in matmul
  X_projected = X_projected @ P_null.T
/var/folders/2l/chysr0_53xx2gf90f24f72jc0000gn/T/ipykernel_7023/1152264484.py:35: RuntimeWarning: overflow encountered in matmul
  X_projected = X_projected @ P_null.T
/var/folders/2l/chysr0_53xx2gf90f24f72jc0000gn/T/ipykernel_7023/1152264484.py:35: RuntimeWarning: invalid value encountered in matmul
  X_projected = X_projected @ P_null.T


INLP Accuracy (Task Y): 0.6753


/var/folders/2l/chysr0_53xx2gf90f24f72jc0000gn/T/ipykernel_7023/1152264484.py:34: RuntimeWarning: divide by zero encountered in matmul
  P_final = P_null @ P_final
/var/folders/2l/chysr0_53xx2gf90f24f72jc0000gn/T/ipykernel_7023/1152264484.py:34: RuntimeWarning: overflow encountered in matmul
  P_final = P_null @ P_final
/var/folders/2l/chysr0_53xx2gf90f24f72jc0000gn/T/ipykernel_7023/1152264484.py:34: RuntimeWarning: invalid value encountered in matmul
  P_final = P_null @ P_final
/var/folders/2l/chysr0_53xx2gf90f24f72jc0000gn/T/ipykernel_7023/1152264484.py:35: RuntimeWarning: divide by zero encountered in matmul
  X_projected = X_projected @ P_null.T
/var/folders/2l/chysr0_53xx2gf90f24f72jc0000gn/T/ipykernel_7023/1152264484.py:35: RuntimeWarning: overflow encountered in matmul
  X_projected = X_projected @ P_null.T
/var/folders/2l/chysr0_53xx2gf90f24f72jc0000gn/T/ipykernel_7023/1152264484.py:35: RuntimeWarning: invalid value encountered in matmul
  X_projected = X_projected @ P_null.T


In [25]:
gender_clf_inlp = LogisticRegression(max_iter=1000)
gender_clf_inlp.fit(X_train_inlp, z_train)
z_pred_inlp = gender_clf_inlp.predict(X_test_inlp)
print(f"INLP Accuracy (Protected Z): {accuracy_score(z_test, z_pred_inlp):.4f}")

INLP Accuracy (Protected Z): 0.5533


/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packag